In [113]:
import os
import pandas as pd
import numpy as np

# 1. Tentukan nama folder tempat file CSV Anda berada
nama_folder = "../Dataset"

# 2. Baca semua file CSV di dalam folder secara otomatis ke dalam dictionary 'tables'
tables = {}
for file in os.listdir(nama_folder):
    if file.endswith('.csv'):
        # Mengambil nama file tanpa ekstensi '.csv' sebagai key (misal: 'kunjungan')
        nama_tabel = os.path.splitext(file)[0] 
        path_file = os.path.join(nama_folder, file)
        
        # Load ke pandas
        tables[nama_tabel] = pd.read_csv(path_file)
        print(f"✔ {nama_tabel:<11} {len(tables[nama_tabel]):>7,} baris | {tables[nama_tabel].shape[1]} kolom")

# Alias untuk tabel yang dipakai di analisis
df_customers = tables["customers"]
df_sellers   = tables["sellers"]
df_products  = tables["products"]
df_orders    = tables["orders"]
df_order_items = tables["order_items"]
df_product_category = tables["product_category_name_translation"]

✔ customers    99,441 baris | 5 kolom
✔ geolocation 1,000,163 baris | 5 kolom
✔ orders       99,441 baris | 8 kolom
✔ order_items 112,650 baris | 7 kolom
✔ order_payments 103,886 baris | 5 kolom
✔ order_reviews  99,224 baris | 7 kolom
✔ products     32,951 baris | 9 kolom
✔ product_category_name_translation      71 baris | 2 kolom
✔ sellers       3,095 baris | 4 kolom


In [114]:
# Membuat dataframe audit
audit = pd.DataFrame({
    "baris": {n: len(t) for n, t in tables.items() if n in ["customers", "order_items", "sellers", "orders","product_category","products"]},
}).assign(
    kolom=lambda d: [tables[n].shape[1] for n in d.index],
    total_null=lambda d: [int(tables[n].isna().sum().sum()) for n in d.index],
)
display(audit)

print("\nNilai kosong per kolom (hanya yang > 0):")
for n in ["Customers", "order_items", "sellers", "orders","product_category","products"]:
    # Proteksi jika file ternyata tidak ada di folder agar kode tidak error
    if n in tables: 
        nulls = tables[n].isna().sum()
        nulls = nulls[nulls > 0]
        if len(nulls):
            print(f"\n[{n}]")
            print(nulls.to_string())


,baris,kolom,total_null
customers,99441,5,0
order_items,112650,7,0
orders,99441,8,4908
products,32951,9,2449
sellers,3095,4,0



Nilai kosong per kolom (hanya yang > 0):

[orders]
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965

[products]
product_id                      1
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2


In [115]:
base_1 = df_orders.merge(df_customers, how='left', on='customer_id')
base_1.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


In [116]:
base_2 = df_order_items.merge(df_sellers, how='left', on='seller_id')
base_2.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,seller_zip_code_prefix,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,27277,volta redonda,SP
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,3471,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,37564,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,14403,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,87900,loanda,PR


In [117]:
main_base_geo_and_loc = base_2.merge(base_1, how="left", on="order_id").assign(is_same_state=lambda d: d['customer_state'] == d['seller_state'])
main_base_geo_and_loc.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,seller_zip_code_prefix,seller_city,seller_state,...,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,is_same_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,27277,volta redonda,SP,...,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29 00:00:00,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,False
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,3471,sao paulo,SP,...,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15 00:00:00,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP,True
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,37564,borda da mata,MG,...,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05 00:00:00,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG,True
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,14403,franca,SP,...,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20 00:00:00,af861d436cfc08b2c2ddefd0ba074622,12952,atibaia,SP,True
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,87900,loanda,PR,...,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17 00:00:00,64b576fb70d441e8f1b2d7d446e483c5,13226,varzea paulista,SP,False


In [118]:
main_base_spec_products = main_base_geo_and_loc.merge(df_products, how="left", on="product_id").assign(product_volume_cm3=lambda d: d['product_length_cm'] * d['product_height_cm'] * d['product_width_cm']).merge(df_product_category[['product_category_name', 'product_category_name_english']], how="left", on="product_category_name")


In [119]:
main_base_spec_products.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,seller_zip_code_prefix,seller_city,seller_state,...,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_volume_cm3,product_category_name_english
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,27277,volta redonda,SP,...,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,3528.0,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,3471,sao paulo,SP,...,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,60000.0,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,37564,borda da mata,MG,...,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,14157.0,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,14403,franca,SP,...,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,2400.0,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,87900,loanda,PR,...,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,42000.0,garden_tools


In [120]:
main_base_spec_products['order_purchase_timestamp'] = pd.to_datetime(main_base_spec_products['order_purchase_timestamp'])
main_base_spec_products['purchase_dayofweek'] = main_base_spec_products['order_purchase_timestamp'].dt.dayofweek
main_base_spec_products['purchase_hour'] = main_base_spec_products['order_purchase_timestamp'].dt.hour

In [121]:
main_base_spec_products = main_base_spec_products.assign(is_late=lambda d: d['order_delivered_customer_date'] > d['order_estimated_delivery_date'])

In [122]:
main_base_spec_products.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,seller_zip_code_prefix,seller_city,seller_state,...,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_volume_cm3,product_category_name_english,purchase_dayofweek,purchase_hour,is_late
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,27277,volta redonda,SP,...,4.0,650.0,28.0,9.0,14.0,3528.0,cool_stuff,2,8,False
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,3471,sao paulo,SP,...,2.0,30000.0,50.0,30.0,40.0,60000.0,pet_shop,2,10,False
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,37564,borda da mata,MG,...,2.0,3050.0,33.0,13.0,33.0,14157.0,furniture_decor,6,14,False
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,14403,franca,SP,...,1.0,200.0,16.0,10.0,15.0,2400.0,perfumery,2,10,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,87900,loanda,PR,...,1.0,3750.0,35.0,40.0,30.0,42000.0,garden_tools,5,13,False


In [123]:
main_df = main_base_spec_products[[
    'customer_state', 'seller_state', 'is_same_state', 'freight_value', 
    'product_weight_g', 'product_volume_cm3', 'price', 'product_category_name_english', 
    'purchase_dayofweek', 'purchase_hour', 'order_approved_at', 'shipping_limit_date', # <- Tambahkan ini
    'order_delivered_carrier_date', 'order_delivered_customer_date', 'is_late'
]].copy()

# Konversi juga ke datetime
datetime_cols = ['order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'shipping_limit_date']
for col in datetime_cols:
    main_df[col] = pd.to_datetime(main_df[col], errors='coerce')

main_df.head()


,customer_state,seller_state,is_same_state,freight_value,product_weight_g,product_volume_cm3,price,product_category_name_english,purchase_dayofweek,purchase_hour,order_approved_at,shipping_limit_date,order_delivered_carrier_date,order_delivered_customer_date,is_late
0,RJ,SP,False,13.29,650.0,3528.0,58.90,cool_stuff,2,8,2017-09-13 09:45:35,2017-09-19 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,False
1,SP,SP,True,19.93,30000.0,60000.0,239.90,pet_shop,2,10,2017-04-26 11:05:13,2017-05-03 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,False
2,MG,MG,True,17.87,3050.0,14157.0,199.00,furniture_decor,6,14,2018-01-14 14:48:30,2018-01-18 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,False
3,SP,SP,True,12.79,200.0,2400.0,12.99,perfumery,2,10,2018-08-08 10:10:18,2018-08-15 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,False
4,SP,PR,False,18.14,3750.0,42000.0,199.90,garden_tools,5,13,2017-02-04 14:10:13,2017-02-13 13:57:51,2017-02-16 09:46:09,2017-03-01 16:42:31,False


Preprocessing

In [124]:
main_df.isnull().sum()

customer_state                      0
seller_state                        0
is_same_state                       0
freight_value                       0
product_weight_g                   19
product_volume_cm3                 19
price                               0
product_category_name_english    1629
purchase_dayofweek                  0
purchase_hour                       0
order_approved_at                  15
shipping_limit_date                 0
order_delivered_carrier_date     1194
order_delivered_customer_date    2454
is_late                             0
dtype: int64

In [125]:
main_df.dropna(subset=['product_weight_g', 'product_volume_cm3', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_delivered_carrier_date'], inplace=True)
main_df['product_category_name_english'] = main_df['product_category_name_english'].fillna('unknown')
main_df.isnull().sum()

customer_state                   0
seller_state                     0
is_same_state                    0
freight_value                    0
product_weight_g                 0
product_volume_cm3               0
price                            0
product_category_name_english    0
purchase_dayofweek               0
purchase_hour                    0
order_approved_at                0
shipping_limit_date              0
order_delivered_carrier_date     0
order_delivered_customer_date    0
is_late                          0
dtype: int64

In [126]:
main_df.duplicated().sum()

10216

In [127]:
main_df.drop_duplicates(inplace=True)

In [128]:
main_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 99945 entries, 0 to 112649
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   customer_state                 99945 non-null  object        
 1   seller_state                   99945 non-null  object        
 2   is_same_state                  99945 non-null  bool          
 3   freight_value                  99945 non-null  float64       
 4   product_weight_g               99945 non-null  float64       
 5   product_volume_cm3             99945 non-null  float64       
 6   price                          99945 non-null  float64       
 7   product_category_name_english  99945 non-null  object        
 8   purchase_dayofweek             99945 non-null  int64         
 9   purchase_hour                  99945 non-null  int64         
 10  order_approved_at              99945 non-null  datetime64[ns]
 11  shipping_limit

In [129]:
main_df.describe()

,freight_value,product_weight_g,product_volume_cm3,price,purchase_dayofweek,purchase_hour
count,99945.000000,99945.000000,99945.000000,99945.000000,99945.000000,99945.000000
mean,20.081584,2088.513472,15104.268688,123.889960,2.758107,14.777638
std,15.793964,3741.629289,23245.030960,187.755629,1.967986,5.330318
min,0.000000,0.000000,168.000000,0.850000,0.000000,0.000000
25%,13.140000,300.000000,2816.000000,40.000000,1.000000,11.000000
50%,16.320000,700.000000,6336.000000,78.450000,3.000000,15.000000
75%,21.200000,1800.000000,18000.000000,139.000000,4.000000,19.000000
max,409.680000,40425.000000,296208.000000,6735.000000,6.000000,23.000000


In [157]:
# ==========================================
# 1. CLEANING & RE-INITIALIZATION
# ==========================================
df_induk_bersih = main_base_spec_products.loc[:, ~main_base_spec_products.columns.duplicated()]

# [TAMBAH] hanya pesanan yang sudah sampai (yang belum sampai tidak punya label valid)
df_induk_bersih = df_induk_bersih[df_induk_bersih['order_status'] == 'delivered']

main_df = df_induk_bersih[[
    'order_id',   # [TAMBAH] untuk split per order
    'customer_state', 'seller_state', 'is_same_state', 'freight_value',
    'product_weight_g', 'product_volume_cm3', 'price',
    'product_category_name_english', 'purchase_dayofweek', 'purchase_hour',
    'order_approved_at', 'order_delivered_carrier_date',
    'order_estimated_delivery_date', 'order_delivered_customer_date',  # [TAMBAH] kolom terakhir
    'shipping_limit_date', 'is_late'
]].copy()

# [UBAH] tambahkan 'order_delivered_customer_date' ke list ini
kolom_waktu = ['order_approved_at', 'order_delivered_carrier_date', 'order_estimated_delivery_date',
               'order_delivered_customer_date', 'shipping_limit_date']
main_df.dropna(subset=kolom_waktu + ['product_weight_g', 'product_volume_cm3'], inplace=True)

for col in kolom_waktu:
    main_df[col] = pd.to_datetime(main_df[col])

# ==========================================
# 2. FEATURE ENGINEERING (REAL-TIME FEATURES)
# ==========================================
main_df['estimated_delivery_days'] = (main_df['order_estimated_delivery_date'] - main_df['order_approved_at']).dt.total_seconds() / 86400
main_df['processing_duration'] = (main_df['order_delivered_carrier_date'] - main_df['order_approved_at']).dt.total_seconds() / 86400
main_df['courier_buffer_days'] = main_df['estimated_delivery_days'] - main_df['processing_duration']

# --- FITUR BARU ---
# Berapa hari selisih penyerahan ke kurir dari batas waktu yang ditetapkan (positif = telat)
main_df['seller_delay'] = (main_df['order_delivered_carrier_date'] - main_df['shipping_limit_date']).dt.total_seconds() / 86400
# ------------------

main_df['freight_ratio'] = main_df['freight_value'] / (main_df['price'] + 0.001)
main_df['state_route'] = main_df['customer_state'] + '_' + main_df['seller_state']

main_df['product_category_name_english'] = main_df['product_category_name_english'].fillna('unknown')
main_df.drop_duplicates(inplace=True)

# Buang tanggal mentah dan komponen wilayah lama, pertahankan teks 'state_route' dan 'product_category_name_english'
main_df.drop(columns=kolom_waktu + ['customer_state', 'seller_state'], errors='ignore', inplace=True)

In [158]:
from sklearn.model_selection import GroupShuffleSplit   # [UBAH] import

x = main_df.drop(columns=['is_late', 'order_id'])
y = main_df['is_late']
groups = main_df['order_id']

# [UBAH] train_test_split diganti split per order_id (item satu order tidak terpisah train/test)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr, te = next(gss.split(x, y, groups))
x_train, x_test = x.iloc[tr].copy(), x.iloc[te].copy()
y_train, y_test = y.iloc[tr], y.iloc[te]
g_train = groups.iloc[tr]

In [161]:
# ==========================================
# 4. HISTORICAL TARGET ENCODING (MURNI PREDIKTIF)
# ==========================================
train_full = x_train.copy()
train_full['is_late'] = y_train

import numpy as np
from sklearn.model_selection import GroupKFold

def oof_encode(cat_train, y_tr, groups_tr, cat_test, smooth=20, n_splits=5):
    cat_tr = cat_train.reset_index(drop=True)
    yy = y_tr.astype(int).reset_index(drop=True)
    gg = groups_tr.reset_index(drop=True)
    prior = yy.mean()
    oof = np.full(len(cat_tr), prior)
    for a, b in GroupKFold(n_splits).split(cat_tr, yy, gg):
        s = yy.iloc[a].groupby(cat_tr.iloc[a]).agg(['sum', 'count'])
        enc = (s['sum'] + prior * smooth) / (s['count'] + smooth)
        oof[b] = cat_tr.iloc[b].map(enc).fillna(prior).values
    s = yy.groupby(cat_tr).agg(['sum', 'count'])
    enc = (s['sum'] + prior * smooth) / (s['count'] + smooth)
    return oof, cat_test.map(enc).fillna(prior).values

# [UBAH] menggantikan route_risk_map / category_risk_map / .map(...) lama
x_train['historical_route_risk'], x_test['historical_route_risk'] = oof_encode(
    x_train['state_route'], y_train, g_train, x_test['state_route'])
x_train['historical_category_risk'], x_test['historical_category_risk'] = oof_encode(
    x_train['product_category_name_english'], y_train, g_train, x_test['product_category_name_english'])
# ==========================================
# 5. PENANGANAN KOLOM KATEGORI (MENCEGAH REDUNDANSI)
# ==========================================
x_train['is_train'] = 1
x_test['is_train'] = 0
combined = pd.concat([x_train, x_test])

# HAPUS kolom kategori asli, JANGAN gunakan pd.get_dummies lagi
combined_encoded = combined.drop(columns=['state_route', 'product_category_name_english'])

# Pisahkan kembali ke matriks final yang siap dimasukkan ke algoritma ML
X_train_final = combined_encoded[combined_encoded['is_train'] == 1].drop(columns=['is_train'])
X_test_final = combined_encoded[combined_encoded['is_train'] == 0].drop(columns=['is_train'])

print("--- PIPELINE SUKSES ---")
print(f"Data Training Bersih: {X_train_final.shape[0]} baris, {X_train_final.shape[1]} fitur")
print(f"Data Testing Bersih: {X_test_final.shape[0]} baris, {X_test_final.shape[1]} fitur")

--- PIPELINE SUKSES ---
Data Training Bersih: 79959 baris, 14 fitur
Data Testing Bersih: 19979 baris, 14 fitur


Modeling

In [162]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# 1. Hitung scale_pos_weight yang lebih moderat (akar kuadrat dari rasio asli)
# Rasio asli: 18712 / 1594 ≈ 11.7 -> Akar kuadrat: ≈ 3.42
moderate_ratio = np.sqrt(18712 / 1594)

# 2. Inisialisasi dan latih model XGBoost
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    scale_pos_weight=moderate_ratio,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_final, y_train)

# 3. Ambil nilai probabilitas untuk kelas 1 (Terlambat)
y_probs = xgb_model.predict_proba(X_test_final)[:, 1]

# 4. Evaluasi performa model di berbagai batas ambang (threshold)
thresholds = [0.50, 0.35, 0.25]

for th in thresholds:
    # Mengubah probabilitas menjadi prediktor biner sesuai threshold
    y_pred_custom = (y_probs >= th).astype(int)
    
    print(f"\n================ EVALUASI THRESHOLD: {th} ================")
    print(classification_report(y_test, y_pred_custom, target_names=['Tepat Waktu (0)', 'Terlambat (1)']))


================ EVALUASI THRESHOLD: 0.5 ================
                 precision    recall  f1-score   support

Tepat Waktu (0)       0.94      0.97      0.95     18397
  Terlambat (1)       0.43      0.29      0.35      1582

       accuracy                           0.91     19979
      macro avg       0.69      0.63      0.65     19979
   weighted avg       0.90      0.91      0.91     19979


================ EVALUASI THRESHOLD: 0.35 ================
                 precision    recall  f1-score   support

Tepat Waktu (0)       0.95      0.90      0.93     18397
  Terlambat (1)       0.29      0.46      0.36      1582

       accuracy                           0.87     19979
      macro avg       0.62      0.68      0.64     19979
   weighted avg       0.90      0.87      0.88     19979


================ EVALUASI THRESHOLD: 0.25 ================
                 precision    recall  f1-score   support

Tepat Waktu (0)       0.96      0.78      0.86     18397
  Terlambat (1) 

In [163]:
import pandas as pd

def engineer_features(df):
    df_new = df.copy()
    
    # 1. Eksploitasi Interaksi Logistik
    df_new['route_volume_risk'] = df_new['historical_route_risk'] * df_new['product_volume_cm3']
    df_new['route_weight_risk'] = df_new['historical_route_risk'] * df_new['product_weight_g']
    
    # 2. Penyederhanaan Fitur Waktu
    # Asumsi purchase_dayofweek: 0=Senin, 6=Minggu
    df_new['is_weekend'] = df_new['purchase_dayofweek'].apply(lambda x: 1 if x >= 5 else 0)
    # Asumsi purchase_hour: format 24 jam (malam hari = jam 18 ke atas atau sebelum jam 6)
    df_new['is_night_order'] = df_new['purchase_hour'].apply(lambda x: 1 if (x >= 18 or x < 6) else 0)
    
    # 3. Eliminasi Fitur Redundan
    cols_to_drop = ['is_same_state', 'price', 'freight_ratio', 'estimated_delivery_days', 
                    'purchase_dayofweek', 'purchase_hour']
    df_new = df_new.drop(columns=[col for col in cols_to_drop if col in df_new.columns])
    
    return df_new

# Terapkan fungsi ke data latih dan uji
X_train_engineered = engineer_features(X_train_final)
X_test_engineered = engineer_features(X_test_final)

# Verifikasi perubahan kolom
print("Fitur baru setelah rekayasa:\n", X_train_engineered.columns.tolist())

Fitur baru setelah rekayasa:
 ['freight_value', 'product_weight_g', 'product_volume_cm3', 'processing_duration', 'courier_buffer_days', 'seller_delay', 'historical_route_risk', 'historical_category_risk', 'route_volume_risk', 'route_weight_risk', 'is_weekend', 'is_night_order']


In [165]:
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (classification_report, roc_auc_score,
                             average_precision_score, precision_recall_curve)

# [HAPUS] SMOTETomek (smt = ..., fit_resample) dan X_train_resampled_eng

# [TAMBAH] validasi diambil dari train, per order, untuk early stopping dan threshold
a, b = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=1)
            .split(X_train_engineered, y_train, g_train))
X_fit, X_val = X_train_engineered.iloc[a], X_train_engineered.iloc[b]
y_fit, y_val = y_train.iloc[a], y_train.iloc[b]

# [UBAH] ketidakseimbangan ditangani lewat bobot kelas; num_leaves 70 -> 15
spw = np.sqrt((y_fit == 0).sum() / (y_fit == 1).sum())
lgb_eng = lgb.LGBMClassifier(                      # [UBAH] nama: lgb_smote_eng -> lgb_eng
    n_estimators=3000, learning_rate=0.03, num_leaves=15,
    min_child_samples=100, reg_lambda=5, scale_pos_weight=spw,
    metric='average_precision', random_state=42, verbose=-1)
lgb_eng.fit(X_fit, y_fit, eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(100, verbose=False)])

# [UBAH] threshold dari validasi, bukan .predict() (0,5) dan bukan dari test
pv = lgb_eng.predict_proba(X_val)[:, 1]
prec, rec, thr = precision_recall_curve(y_val, pv)
th = thr[np.argmax(2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-9, None))]

pt = lgb_eng.predict_proba(X_test_engineered)[:, 1]
print(f"ROC-AUC {roc_auc_score(y_test, pt):.3f} | PR-AUC {average_precision_score(y_test, pt):.3f} "
      f"(baseline {y_test.mean():.3f}) | thr {th:.3f}")
print(classification_report(y_test, (pt >= th).astype(int), digits=3))

c:\Users\ASUS\anaconda3\envs\dl_gpu\lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


ROC-AUC 0.794 | PR-AUC 0.383 (baseline 0.079) | thr 0.442
              precision    recall  f1-score   support

       False      0.946     0.944     0.945     18397
        True      0.363     0.369     0.366      1582

    accuracy                          0.899     19979
   macro avg      0.654     0.656     0.655     19979
weighted avg      0.899     0.899     0.899     19979

